# Student Performance in R: GPA Drivers

This notebook studies which behavioral and socioeconomic factors are most associated with GPA in the Student Academic Performance dataset, with a focus on practical interpretation rather than only descriptive reporting.

## Objective

The objective is to explain variation in `overall_gpa` and identify which inputs deserve intervention priority. The working success metric is out-of-sample RMSE from a transparent linear baseline, supported by subgroup summaries and visual inspection.

In [ ]:
library(dplyr)
library(ggplot2)

set.seed(42)

data_path <- "/kaggle/input/student-academic-performance-dataset/students.csv"
if (!file.exists(data_path)) {
  data_path <- "students.csv"
}

students <- read.csv(data_path)
glimpse(students)


## Data Overview

The dataset combines GPA, attendance, study time, sleep, stress, motivation, and family background features. Before modeling, we check scale, central tendency, and missingness so the later findings are easier to interpret and reproduce.

In [ ]:
students %>%
  summarise(
    rows = n(),
    avg_gpa = mean(overall_gpa, na.rm = TRUE),
    avg_attendance = mean(attendance_rate, na.rm = TRUE),
    avg_study_hours = mean(study_hours_per_week, na.rm = TRUE),
    missing_values = sum(is.na(.))
  )


In [ ]:
students %>%
  group_by(parental_education) %>%
  summarise(
    mean_gpa = mean(overall_gpa, na.rm = TRUE),
    mean_attendance = mean(attendance_rate, na.rm = TRUE),
    mean_study_hours = mean(study_hours_per_week, na.rm = TRUE),
    .groups = "drop"
  )


## Exploratory Analysis

A first observation is that attendance and study hours should move with GPA, but the relationship may flatten if stress is high. This plot is useful because it shows both the overall signal and whether subgroup slopes differ in a way that suggests equity or access limitations.

In [ ]:
ggplot(students, aes(x = study_hours_per_week, y = overall_gpa, color = parental_education)) +
  geom_point(alpha = 0.25) +
  geom_smooth(method = "lm", se = FALSE) +
  labs(
    title = "Study Hours vs GPA",
    subtitle = "Segmented by parental education to support fairness-oriented inspection",
    x = "Study hours per week",
    y = "Overall GPA"
  )


## Method

The method uses a simple train/test split and a linear regression baseline. This approach is intentionally transparent: coefficients are easy to inspect, the baseline is reproducible, and any limitation is visible before moving to more complex models.

In [ ]:
train_index <- sample(seq_len(nrow(students)), size = floor(0.8 * nrow(students)))
train <- students[train_index, ]
test <- students[-train_index, ]

gpa_model <- lm(
  overall_gpa ~ study_hours_per_week + attendance_rate + sleep_hours + stress_level + motivation_score,
  data = train
)

test_predictions <- predict(gpa_model, newdata = test)
rmse <- sqrt(mean((test$overall_gpa - test_predictions)^2, na.rm = TRUE))

list(
  rmse = rmse,
  coefficients = coef(summary(gpa_model))
)


In [ ]:
diagnostics <- data.frame(
  actual_gpa = test$overall_gpa,
  predicted_gpa = test_predictions,
  abs_error = abs(test$overall_gpa - test_predictions)
)

ggplot(diagnostics, aes(x = predicted_gpa, y = abs_error)) +
  geom_point(alpha = 0.3, color = "steelblue") +
  geom_smooth(se = FALSE, color = "firebrick") +
  labs(
    title = "Prediction Error by Predicted GPA",
    subtitle = "Used for evaluation and error analysis",
    x = "Predicted GPA",
    y = "Absolute error"
  )


## Evaluation and Findings

The main finding is whether attendance and motivation remain positive after controlling for sleep and stress. An important insight is that higher study hours may help because they proxy for preparation discipline, but the trade-off is that stress can offset some of that gain. A useful limitation to note is that the model is linear and may miss threshold effects, so the hypothesis should be stress-tested with tree-based models later.

## Conclusion and Next Steps

In summary, this notebook provides a reproducible baseline for understanding GPA drivers in R. The recommended next step is to compare subgroup error rates, improve the feature set with interaction terms, and test whether non-linear models materially outperform the interpretable baseline.